# Fine-tuning Language Models with AdvSecureNet

This notebook demonstrates how to fine-tune large language models using the AdvSecureNet LLM fine-tuning framework. The framework supports:

- **Parameter-Efficient Fine-tuning (PEFT)** with LoRA and QLoRA
- **Multiple data sources** including local JSONL files and HuggingFace Hub datasets
- **Flexible configuration** through YAML files or programmatic setup
- **Memory-efficient training** with gradient checkpointing and quantization

We'll walk through the complete process from loading configuration to training and testing the fine-tuned model.

## Import Required Libraries

First, we import the necessary components from the AdvSecureNet LLM fine-tuning module:

- **Configuration classes**: For defining training parameters, data sources, and PEFT settings
- **Training pipeline**: The main `run_training` function that orchestrates the entire process

In [ ]:
from advsecurenet.llm_finetuning.config import load_yaml, Config, TrainConfig, DataConfig, PEFTConfig
from advsecurenet.llm_finetuning.train import run_training
import os

## Load Configuration

We load the training configuration from a YAML file. This configuration defines:

- **Model settings**: Which pre-trained model to use (e.g., GPT-2, Pythia, LLaMA)
- **Training parameters**: Learning rate, batch size, number of steps, mixed precision
- **Data source**: Either local JSONL files or HuggingFace Hub datasets
- **PEFT configuration**: LoRA rank, alpha, target modules, and quantization settings

The configuration system provides type safety and validation to ensure all settings are correct.

In [ ]:
# Load configuration from YAML file
config_path = "../../../advsecurenet/llm_finetuning/configs/finetune.yaml"
config = load_yaml(config_path)
# Display key configuration settings
print(f"Model: {config.train.model_name}")
print(f"Dataset: {config.data.hub_name}")
print(f"Output dir: {config.train.output_dir}")
print(f"PEFT enabled: {config.peft.enabled}")
print(f"Max steps: {config.train.max_steps}")
print(f"Learning rate: {config.train.learning_rate}")
print(f"LoRA rank: {config.peft.r}")

## Run Fine-tuning

Now we execute the complete fine-tuning pipeline. The `run_training` function will:

1. **Set random seeds** for reproducible results
2. **Load the tokenizer** and configure padding tokens
3. **Load and tokenize datasets** with proper formatting (instruction-following, completion, etc.)
4. **Load the base model** with optional quantization for memory efficiency
5. **Apply PEFT adapters** (LoRA) if enabled to reduce trainable parameters
6. **Configure training arguments** from our settings (batch size, learning rate, etc.)
7. **Create and run the Trainer** using HuggingFace's training loop
8. **Save the fine-tuned model** and tokenizer to the output directory

The training process will show progress logs including loss, learning rate, and other metrics.

In [ ]:
print("🚀 Starting training...")
result = run_training(config)
print(f"✅ Training completed! Model saved to: {result['output_dir']}")

## Test the Fine-tuned Model

After training completes, let's load the fine-tuned model and test its text generation capabilities. 

This section demonstrates how to:

1. **Load the saved model and tokenizer** from the output directory
2. **Prepare input prompts** for text generation
3. **Generate responses** using the fine-tuned model with sampling parameters
4. **Compare outputs** to see how the model has adapted to the training data

The generation uses:
- **Temperature (0.7)**: Controls randomness in token selection
- **max_new_tokens (50)**: Limits the length of generated text
- **do_sample (True)**: Enables sampling instead of greedy decoding for more diverse outputs

You can experiment with different prompts to see how well the model has learned from the fine-tuning data.

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    
    # Load the fine-tuned model and tokenizer from the saved directory
    model_path = result['output_dir']
    print(f"Loading model from: {model_path}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    
    # Test with multiple prompts to see the model's behavior
    test_prompts = [
        "What is artificial intelligence?",
        "Explain the concept of machine learning.",
        "How does deep learning work?"
    ]
    
    print("\n" + "="*60)
    print("TESTING FINE-TUNED MODEL GENERATION")
    print("="*60)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n--- Test {i} ---")
        print(f"Prompt: {prompt}")
        
        # Tokenize the input prompt
        inputs = tokenizer(prompt, return_tensors="pt")
        
        # Generate response using the fine-tuned model
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,           # Generate up to 50 new tokens
                do_sample=True,              # Use sampling for variety
                temperature=0.7,             # Control randomness
                top_p=0.9,                   # Nucleus sampling
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1       # Reduce repetitive outputs
            )
        
        # Decode and display the generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = generated_text[len(prompt):].strip()  # Extract only the new text
        print(f"Response: {response}")
        print("-" * 40)
    
    print("\n✅ Model testing completed successfully!")
    
except Exception as e:
    print(f"❌ Model testing failed: {e}")
    print("This might happen if:")
    print("- Training was interrupted or failed")
    print("- Output directory doesn't contain valid model files")
    print("- Memory issues loading the model")

## Configuration Options

The AdvSecureNet LLM fine-tuning framework offers extensive customization options:

### Training Parameters
- **Learning Rate**: Controls how fast the model adapts (typically 1e-4 to 5e-4 for LoRA)
- **Batch Size**: Memory vs. speed tradeoff (adjust `per_device_train_batch_size`)
- **Gradient Accumulation**: Simulate larger batches with limited memory
- **Mixed Precision**: Use bf16/fp16 for faster training on modern GPUs

### PEFT (Parameter-Efficient Fine-Tuning)
- **LoRA Rank (r)**: Higher rank = more adaptation capacity but more parameters
- **LoRA Alpha**: Scaling factor for LoRA adaptations
- **Target Modules**: Which transformer layers to adapt (attention, MLP, etc.)
- **Quantization**: Use QLoRA for 4-bit training to save memory

### Data Sources
- **Local JSONL**: Custom datasets in JSON Lines format
- **HuggingFace Hub**: Access thousands of public datasets
- **Field Mapping**: Configure prompt/response fields for instruction tuning

You can modify the YAML configuration file or create configs programmatically for different use cases.